# RTMDet-tiny reproduction

In [1]:
import subprocess

gpu = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(gpu)
if not gpu.startswith("Tesla T4,"):
    raise RuntimeError("select a Tesla T4 runtime for this reproduction")

Tesla T4, 580.82.07, 15360 MiB


In [2]:
import os
from pathlib import Path

installer = Path("/tmp/uv-install.sh")
subprocess.run(
    [
        "curl",
        "-LsSf",
        "https://astral.sh/uv/0.12.23/install.sh",
        "-o",
        str(installer),
    ],
    check=True,
)
env = os.environ.copy()
env["UV_UNMANAGED_INSTALL"] = "/usr/local/bin"
subprocess.run(["sh", str(installer)], check=True, env=env)
uv_version = subprocess.run(
    ["uv", "--version"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(uv_version)
if not uv_version.startswith("uv 0.12.23 "):
    raise RuntimeError(f"unexpected uv version: {uv_version}")

uv 0.12.23 (x86_64-unknown-linux-gnu)


In [3]:
import shutil

repository = "https://github.com/aekn/smallpotato.git"
revision = "837b9dd66e6c0ecf36f5c33ed03d99be6096ce06"

repo = Path("/content/smallpotato")
if repo.exists():
    shutil.rmtree(repo)

subprocess.run(["git", "init", "-q", str(repo)], check=True)
subprocess.run(
    ["git", "-C", str(repo), "remote", "add", "origin", repository],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(repo),
        "fetch",
        "-q",
        "--depth=1",
        "origin",
        revision,
    ],
    check=True,
)
subprocess.run(
    ["git", "-C", str(repo), "checkout", "-q", "--detach", "FETCH_HEAD"],
    check=True,
)

actual = subprocess.run(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != revision:
    raise RuntimeError(f"expected {revision}, got {actual}")

status = subprocess.run(
    ["git", "-C", str(repo), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout
if status:
    raise RuntimeError("repository checkout is dirty")

os.chdir(repo)
print(f"{repo} @ {actual}")

/content/smallpotato @ 837b9dd66e6c0ecf36f5c33ed03d99be6096ce06


In [4]:
subprocess.run(["uv", "sync", "--locked"], cwd=repo, check=True)

CompletedProcess(args=['uv', 'sync', '--locked'], returncode=0)

In [5]:
process = subprocess.Popen(
    [
        "uv",
        "run",
        "python",
        "-m",
        "reproductions.rtmdet.reproduce",
    ],
    cwd=repo,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

returncode = process.wait()
print(f"exit status {returncode}")

smallpotato - reproduce rtmdet

model       rtmdet-tiny
dataset     coco/val2017
run         /content/smallpotato/runs/rtmdet/20261007T213807Z

sync        environment
device      Tesla T4
check       environment                 ok
prepare     inputs
check       source                      ok
check       data                        ok
check       checkpoint                  ok
test          50/1000    5.0%   eta 6m26s
test         100/1000   10.0%   eta 5m45s
test         150/1000   15.0%   eta 5m25s
test         200/1000   20.0%   eta 5m01s
test         250/1000   25.0%   eta 4m41s
test         300/1000   30.0%   eta 4m22s
test         350/1000   35.0%   eta 4m02s
test         400/1000   40.0%   eta 3m43s
test         450/1000   45.0%   eta 3m24s
test         500/1000   50.0%   eta 3m05s
test         550/1000   55.0%   eta 2m47s
test         600/1000   60.0%   eta 2m28s
test         650/1000   65.0%   eta 2m09s
test         700/1000   70.0%   eta 1m51s
test         750/1000   75.0%   

In [6]:
import json

runs = repo / "runs" / "rtmdet"
run_dir = max(
    (path for path in runs.iterdir() if path.is_dir()),
    key=lambda path: path.name,
)
summary = json.loads((run_dir / "summary.json").read_text())
print(run_dir)
print(json.dumps(summary, indent=2))

/content/smallpotato/runs/rtmdet/20261007T213807Z
{
  "artifacts": {
    "log": "run.log",
    "predictions": {
      "path": "pred.bbox.json",
      "sha256": "1dc864d877662a6544f6fe218862a59e3f7142a0c3c17464dde9910e1372ae83",
      "size": 234092583
    },
    "provenance": "provenance.json",
    "runtime_config": "runtime.py",
    "upstream": "upstream"
  },
  "durations_seconds": {
    "coco_accumulate": 23.816097598000056,
    "coco_evaluate": 93.71807955099996,
    "coco_load": 12.39050049299999,
    "coco_summarize": 2.5299386539999205,
    "coco_total": 132.45461629599993,
    "environment": 70.46254671,
    "inputs": 88.04922755200002,
    "test": 414.028092108,
    "total": 705.789464232
  },
  "expected": {
    "absolute_tolerance": 0.002,
    "metrics": {
      "ap": 0.4112,
      "ap50": 0.5787,
      "ap75": 0.4472,
      "ap_large": 0.583,
      "ap_medium": 0.4551,
      "ap_small": 0.2099,
      "ar1": 0.334,
      "ar10": 0.5536,
      "ar100": 0.6053,
      "ar_large

In [7]:
from google.colab import files

archive_base = Path("/content") / f"rtmdet-{run_dir.name}"
run_archive = shutil.make_archive(
    str(archive_base),
    "gztar",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)
print(run_archive)
files.download(run_archive)

/content/rtmdet-20261007T213807Z.tar.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>